<a href="https://colab.research.google.com/github/ibnu-ahmedin/afaan-oromo-hybrid-sentiment-analysis/blob/main/Lexicon_Construction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================================
#LEXICON EXPANSION FROM DATASET
# ==========================================================

import pandas as pd
import re
from collections import Counter, defaultdict

# =====================================
# 1️⃣ LOAD DATASET
# =====================================
df = pd.read_csv('/content/Full_Corpus_10005.csv')

df = df.dropna(subset=['Text', 'Sentiment'])
df['Sentiment'] = df['Sentiment'].str.strip().str.capitalize()

print("Dataset size:", len(df))

# =====================================
# 2️⃣ TOKENIZATION
# =====================================
def normalize_word(word):
    word = re.sub(r'(.)\1{2,}', r'\1\1', word)
    word = word.replace("’", "'")
    return word

def tokenize(text):
    words = re.findall(r"\b[\w']+\b", str(text).lower())
    return [normalize_word(w) for w in words]

# =====================================
# 3️⃣ COUNT WORD FREQUENCIES BY CLASS
# =====================================
word_stats = defaultdict(lambda: {"Positive":0, "Negative":0, "Neutral":0})

for _, row in df.iterrows():
    tokens = set(tokenize(row["Text"]))  # unique words per text
    label = row["Sentiment"]

    for word in tokens:
        word_stats[word][label] += 1

# =====================================
# 4️⃣ CONVERT TO DATAFRAME
# =====================================
records = []

for word, counts in word_stats.items():
    total = sum(counts.values())

    if total < 5:  # 🔥 filter rare words
        continue

    pos = counts["Positive"]
    neg = counts["Negative"]
    neu = counts["Neutral"]

    # =====================================
    # 🔥 POLARITY SCORE
    # =====================================
    score = (pos - neg) / total

    records.append({
        "Word": word,
        "Positive": pos,
        "Negative": neg,
        "Neutral": neu,
        "Total": total,
        "Score": score
    })

stats_df = pd.DataFrame(records)

# =====================================
# 5️⃣ SELECT STRONG WORDS ONLY
# =====================================
strong_words = stats_df[
    (stats_df["Score"].abs() >= 0.3) &
    (stats_df["Total"] >= 5)
]

print("Candidate words:", len(strong_words))

# =====================================
# 6️⃣ ASSIGN CATEGORY + FINAL SCORE
# =====================================
def assign_category(row):
    if row["Score"] > 0:
        return "positive"
    else:
        return "negative"

strong_words["Category"] = strong_words.apply(assign_category, axis=1)

# scale score to [-3, 3]
strong_words["FinalScore"] = strong_words["Score"] * 3

# =====================================
# 7️⃣ LOAD EXISTING LEXICON
# =====================================
lexicon_df = pd.read_csv('/content/Final_Lexicon_v10.csv')

existing_words = set(lexicon_df["Word"].str.lower())

# =====================================
# 8️⃣ REMOVE EXISTING WORDS
# =====================================
new_words = strong_words[
    ~strong_words["Word"].isin(existing_words)
]

print("New words to add:", len(new_words))

# =====================================
# 9️⃣ CREATE NEW LEXICON ENTRIES
# =====================================
auto_entries = new_words[["Word", "Category", "FinalScore"]]
auto_entries.columns = ["Word", "Category", "Score"]

# =====================================
# 🔟 MERGE WITH LEXICON
# =====================================
expanded_lexicon = pd.concat([lexicon_df, auto_entries], ignore_index=True)

expanded_lexicon = expanded_lexicon.drop_duplicates(subset=["Word"])
expanded_lexicon = expanded_lexicon.reset_index(drop=True)

# =====================================
# 1️⃣1️⃣ SAVE FINAL VERSION
# =====================================
expanded_lexicon.to_csv('/content/Final_Lexicon_v11_auto.csv', index=False)

print("\n🔥 FINAL AUTO-EXPANDED LEXICON")
print("New size:", len(expanded_lexicon))

print("\nSample new entries:")
print(auto_entries.sample(10))

Dataset size: 10005
Candidate words: 1284
New words to add: 710

🔥 FINAL AUTO-EXPANDED LEXICON
New size: 2052

Sample new entries:
           Word  Category     Score
1390    fiilmii  positive  2.000000
331         bar  negative -1.307692
241          my  positive  1.625000
2358    banamuu  negative -1.200000
1965     jibbuu  negative -1.500000
649      wantii  negative -1.071429
2356  barbaanna  negative -1.200000
800     hamilee  negative -1.500000
1196     ammati  negative -1.800000
656       horii  negative -0.923077


/tmp/ipykernel_4640/452019353.py:93: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  strong_words["Category"] = strong_words.apply(assign_category, axis=1)
/tmp/ipykernel_4640/452019353.py:96: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  strong_words["FinalScore"] = strong_words["Score"] * 3


In [ ]:
# ==========================================================
# LEXICON COVERAGE ON THE ALIGNED TEST SET (1,001 samples)
# Used for Section 4.2 of the thesis
# ==========================================================

import pandas as pd
import re

# ==========================================
# 1. Load the test set (same as all other models)
# ==========================================
test_df = pd.read_csv('/content/test.csv')

# ==========================================
# 2. Exact cleaning function used in your latest experiments
# ==========================================
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)          # remove URLs
    text = re.sub(r'@\w+', '@user', text)               # normalize mentions
    text = re.sub(r'[\U0001F600-\U0001FFFF]', '', text) # remove emojis
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)          # collapse repeated chars
    text = re.sub(r"[^\w\s']", '', text)                # remove punctuation (keep apostrophe)
    return ' '.join(text.split())

# Apply cleaning
test_df['clean_text'] = test_df['Text'].apply(clean_text)

# ==========================================
# 3. Load the lexicon (exactly as in hybrid & rule-based)
# ==========================================
lexicon_df = pd.read_csv('/content/Final_Lexicon_v11_auto.csv')
lexicon_df['Word'] = lexicon_df['Word'].str.lower().str.strip()

# Get set of all lexicon words (polarity + phrases + negations, etc.)
lexicon_words = set(lexicon_df['Word'].unique())

# ==========================================
# 4. Get unique tokens from the cleaned test set
# ==========================================
all_test_tokens = set()
for text in test_df['clean_text']:
    tokens = text.split()          # same simple split as your rule-based function
    all_test_tokens.update(tokens)

# ==========================================
# 5. Compute coverage
# ==========================================
covered = len(all_test_tokens & lexicon_words)
total_unique = len(all_test_tokens)
coverage_pct = (covered / total_unique) * 100 if total_unique > 0 else 0

print("✅ Lexicon coverage calculation complete")
print(f"Unique tokens in cleaned test set : {total_unique:,}")
print(f"Tokens covered by lexicon         : {covered:,}")
print(f"Coverage percentage               : {coverage_pct:.1f}%")


✅ Lexicon coverage calculation complete
Unique tokens in cleaned test set : 4,440
Tokens covered by lexicon         : 1,009
Coverage percentage               : 22.7%
